# Detecting Sparsity, Dependency and Invertibility in Expression Trees

The module `FFDep` in PyMC++ determines the __sparsity pattern__ and __type of dependencies__ in expression trees with respect to participating variables. The detected dependencies include:
- Linear (`L`) for a variable that only participates in linear terms
- Quadratic (`Q`) for a variable that only participates in (linear or) quadratic terms
- Polynomial (`P`) for a variable that only participates in (linear, quadratic or) polynomial terms
- Rational (`R`) for a variable that only participates in (linear, quadratic, polynomial or) rational terms
- Nonlinear (`N`) for a variable that only participates in (general) nonlinear terms

Similarly, the module `FFInv` in PyMC++ determines the __invertibility__ of given expression trees with respect to their participating variables. Possible outcomes of such invertibility analysis include:
- Linear (`L`) for a variable that only participate linearly
- Separably linear (`S`) for a variable that only participates in linear or multilinear terms
- Separably nonlinear (`N`) for a variable that participates in nonlinear terms are invertible (as specified in options)
- Undetermined (`U`) for a variable whose invertibility cannot be established or is not possible

The propagation can either be done directly through operator overloading or via the evaluation of a DAG of the expression. Next, we illustrate these capabilities for the factorable function ${\bf f}:\mathbb{R}^4\to\mathbb{R}^2$ defined by
$${\bf f}(x_0,x_1,x_2,x_3) \coloneqq \left(\begin{array}{c} \displaystyle x_2 x_3+\frac{x_0}{x_2}\\x_0^2[\exp(x_2)+x_3]+x_1 \end{array}\right)$$


In [1]:
import pymcpp

In [2]:
def f( x ):
    return [ x[2] * x[3] + x[0] / x[2],
             pymcpp.sqr( x[0] ) * ( pymcpp.exp( x[2] ) + x[3] ) + x[1] ]

## Dependency Detection in PyMC++

First, we define the variables $x_0\ldots,x_3$ as:

In [3]:
NX = 4;
XDep = [ pymcpp.FFDep().indep(i) for i in range(NX) ]
print( XDep )

[{ 0L }, { 1L }, { 2L }, { 3L }]


This defines a list of `pymcpp.FFDep` variables, initialized with the corresponding index $i=0\ldots 3$ and thus showing linear dependence (`L`) with respect to their respective index.

We can then simply propagate these dependencies through the function $\bf f$ using operator overloading:

In [4]:
FDep = f( XDep )
print( FDep )

[{ 0R 2R 3Q }, { 0N 1L 2N 3P }]


The results indicate that variables $x_0$, $x_2$ and $x_3$ participate in $f_0$, but not $x_1$; and that $x_0$ and $x_2$ have rational (`R`) dependence  terms, while $x_3$ has quadratic (`Q`) dependence. Likewise, all four variables $x_0\ldots x_3$ participate in $f_1$, with $x_0$ and $x_2$ showing nonlinear (`N`), $x_3$ polynomial (`P`) dependence, and $x_1$ linear (`L`) dependence.

The member functions `pymcpp.FFDep.dep` can be used to retrieve the resulting dependencies in a dictionary or to query the dependency on  a specific variable: 

In [5]:
print( FDep[0].dep(2) )
print( FDep[0].dep(2) )

(True, <TYPE.R: 3>)
(True, <TYPE.R: 3>)


The dependencies can also be propagated through a DAG of $\bf f$:

In [6]:
DAG  = pymcpp.FFGraph()
X = DAG.add_vars( NX, "x" )
F = f( X )
print( "f0(x) = ", F[0].str(), "\nf1(x) = ", F[1].str() )

FDep = DAG.eval( F, X, XDep )
print( FDep )

f0(x) =  x2 * x3 + x0 / x2 
f1(x) =  x1 + SQR( x0 ) * ( x3 + EXP( x2 ) )
[{ 0R 2R 3Q }, { 0N 1L 2N 3P }]


## Invertibility Detection in PyMC++

Invertibility analysis proceeds similarly, by first defining the variables $x_0\ldots,x_3$ as:

In [7]:
NX = 4;
XInv = [ pymcpp.FFInv().indep(i) for i in range(NX) ]
print( XInv )

[{ 0L }, { 1L }, { 2L }, { 3L }]


We then simply propagate these variables through the function $\bf f$ using operator overloading:

In [8]:
FInv = f( XInv )
print( FInv )

[{ 0S 2U 3S }, { 0U 1L 2N 3S }]


These results indicate that $x_0$, $x_2$ and $x_3$ participate in $f_0$, but not $x_1$; and that $x_0$ and $x_3$ are separably linear (`S`), but the invertibility of $x_2$ is undetermined (`U`), which is due to multiple occurrences. 

Likewise, all four variables $x_0,\ldots,x_3$ participate in $f_1$; the invertibility of $x_0$ is undetermined (this is since `IPOW` is not allowed as invertible in the option set `FFInv::Options::INVOP` by default), while $x_1$ participates linearly (`L`) and is thus invertible, $x_3$ is separably linear (`S`), and $x_2$ is recognized as nonlinearly invertible (`N`) (this is since `EXP` is allowed as invertible in the option set `FFInv::Options::INVOP` by default). If we were to allow the square term to be invertible, the invertibility of $x_0$ in $f_1$ would change to `N`:

In [9]:
pymcpp.FFInv.options.reset()
pymcpp.FFInv.options.INVOP.add( pymcpp.FFInv.options.IPOW )
print( pymcpp.FFInv.options.INVOP )

FInv = f( XInv )
print( FInv )

{INV, SQRT, EXP, LOG, IPOW, RPOW}
[{ 0S 2U 3S }, { 0N 1L 2N 3S }]


The dependencies can be propagated through a DAG of $\bf f$ as well:

In [10]:
FInv = DAG.eval( F, X, XInv )
print( FInv )

[{ 0S 2U 3S }, { 0N 1L 2N 3S }]
